# Evaluation & Analysis — Deep Models (R(2+1)D-18, CNN-GRU, TSM)

Threshold calibration, final results, plots, and Grad-CAM saliency analysis for the three deep models trained in the training notebook (checkpoints in `outputs_all_3_models_v2/`).

**Stages:**
1. Setup, splits, model builders
2. PR curves + F1-vs-threshold curves (saves `curves_*.npz`)
3. **Dual-threshold selection** — primary (precision ≥ 0.60) and secondary (recall ≥ 0.70) on validation only
4. **Final results table** → dissertation Table 4.2 (CSV + PNG)
5. **Grad-CAM** with diagnostic case selection (confident TP / FP / hardest FN / cleanest TN)

> Thresholds are frozen on validation, then applied unchanged to the UTA-RLDD, YawDD and DROZY test sets.

---
## 1. Setup, Splits & Builders

In [ ]:
from google.colab import drive
drive.mount("/content/drive", force_remount=False)

In [ ]:
from google.colab import drive
drive.mount("/content/drive", force_remount=False)

import os
import pandas as pd
from collections import Counter
from pathlib import Path

SPLITS_DIR   = "/content/drive/MyDrive/ES327_Drowsiness/splits"
CLIPS_DRIVE  = "/content/drive/MyDrive/ES327_Drowsiness/Clips"
CLIPS_LOCAL  = "/content/Clips"

SAVE_ROOT = "/content/drive/MyDrive/ES327_Drowsiness/outputs_all_3_models"  # where your best.pt live
print("SAVE_ROOT:", SAVE_ROOT)

In [ ]:
# Use the final v2 checkpoints
SAVE_ROOT = "/content/drive/MyDrive/ES327_Drowsiness/outputs_all_3_models_v2"
print("SAVE_ROOT:", SAVE_ROOT)

In [ ]:
import torch
NUM_CLASSES = 2
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device, "| NUM_CLASSES:", NUM_CLASSES)

In [ ]:
import os
import pandas as pd
from collections import Counter

SPLITS_DIR = "/content/drive/MyDrive/ES327_Drowsiness/splits"

UTA_TRAIN  = os.path.join(SPLITS_DIR, "uta_train.csv")
UTA_VAL    = os.path.join(SPLITS_DIR, "uta_val.csv")
UTA_TEST   = os.path.join(SPLITS_DIR, "uta_test.csv")
YAW_TRAIN  = os.path.join(SPLITS_DIR, "yawdd_train.csv")
YAW_VAL    = os.path.join(SPLITS_DIR, "yawdd_val.csv")
DROZY_TEST = os.path.join(SPLITS_DIR, "drozy_test.csv")

def load_split_csv(path):
    df = pd.read_csv(path)
    df = df.dropna(subset=["clip_path", "label"]).copy()
    df["label"] = df["label"].astype(int)
    return df

def namespace_subjects(df, default_ds):
    df = df.copy()
    if "dataset" in df.columns:
        ds = df["dataset"].fillna(default_ds).astype(str).str.lower()
    else:
        ds = pd.Series([default_ds]*len(df), index=df.index)
    df["subject_id"] = ds + "_" + df["subject_id"].astype(str)
    return df

# Load
uta_train  = namespace_subjects(load_split_csv(UTA_TRAIN),  "uta")
uta_val    = namespace_subjects(load_split_csv(UTA_VAL),    "uta")
uta_test   = namespace_subjects(load_split_csv(UTA_TEST),   "uta")
yaw_train  = namespace_subjects(load_split_csv(YAW_TRAIN),  "yawdd")
yaw_val    = namespace_subjects(load_split_csv(YAW_VAL),    "yawdd")
drozy_test = namespace_subjects(load_split_csv(DROZY_TEST), "drozy")

# Combine
train_df    = pd.concat([uta_train, yaw_train], ignore_index=True)
val_df      = pd.concat([uta_val,   yaw_val],   ignore_index=True)
test_df     = uta_test.copy()
external_df = drozy_test.copy()

print("Train:", len(train_df), dict(Counter(train_df["label"])))
print("Val:", len(val_df), dict(Counter(val_df["label"])))
print("Test:", len(test_df), dict(Counter(test_df["label"])))
print("External:", len(external_df), dict(Counter(external_df["label"])))

In [ ]:
CLIPS_DRIVE_P = Path(CLIPS_DRIVE)
CLIPS_LOCAL_P = Path(CLIPS_LOCAL)

def map_to_local_df(df):
    df = df.copy()
    def _map(p):
        p = str(p)
        pref = str(CLIPS_DRIVE_P) + "/"
        if p.startswith(pref):
            rel = p[len(pref):]
            local = CLIPS_LOCAL_P / rel
            if local.exists():
                return str(local)
        return p
    df["clip_path_mapped"] = df["clip_path"].astype(str).apply(_map)
    return df

train_df_m    = map_to_local_df(train_df)
val_df_m      = map_to_local_df(val_df)
test_df_m     = map_to_local_df(test_df)
external_df_m = map_to_local_df(external_df)

print("Mapped counts:",
      "train", len(train_df_m),
      "val", len(val_df_m),
      "test", len(test_df_m),
      "ext", len(external_df_m))

print("Example mapped path:", train_df_m["clip_path_mapped"].iloc[0])

In [ ]:
train_df_m    = map_to_local_df(train_df)
val_df_m      = map_to_local_df(val_df)
test_df_m     = map_to_local_df(test_df)
external_df_m = map_to_local_df(external_df)

print("Mapped OK.")

In [ ]:
import torchvision
import torch.nn as nn

NUM_CLASSES = 2

# 3D-CNN: 3D ResNet-18
def build_r3d18(num_classes=2):
    m = torchvision.models.video.r3d_18(weights=None)
    m.fc = nn.Linear(m.fc.in_features, num_classes)
    return m

# CNN-GRU: ResNet18 per-frame + GRU
class ResNetGRU(nn.Module):
    def __init__(self, hidden=256, num_classes=2):
        super().__init__()
        base = torchvision.models.resnet18(weights=None)
        self.backbone = nn.Sequential(*list(base.children())[:-1])
        self.feat_dim = 512
        self.gru = nn.GRU(self.feat_dim, hidden, batch_first=True)
        self.fc = nn.Linear(hidden, num_classes)

    def forward(self, x):  # (B,T,C,H,W)
        B,T,C,H,W = x.shape
        x = x.reshape(B*T, C, H, W)
        f = self.backbone(x).flatten(1)
        f = f.reshape(B, T, self.feat_dim)
        out, _ = self.gru(f)
        return self.fc(out[:, -1, :])

def build_cnn_gru(num_classes=2):
    return ResNetGRU(hidden=256, num_classes=num_classes)

# TSM-style temporal average baseline
class ResNetTemporalAvg(nn.Module):
    def __init__(self, num_classes=2):
        super().__init__()
        base = torchvision.models.resnet18(weights=None)
        self.backbone = nn.Sequential(*list(base.children())[:-1])
        self.fc = nn.Linear(512, num_classes)

    def forward(self, x):  # (B,C,T,H,W)
        B,C,T,H,W = x.shape
        x = x.permute(0,2,1,3,4).contiguous().view(B*T, C, H, W)
        f = self.backbone(x).flatten(1).view(B, T, 512).mean(dim=1)
        return self.fc(f)

def build_tsm_fast(num_classes=2):
    return ResNetTemporalAvg(num_classes=num_classes)

print("✅ Builders defined.")

---
## 2. PR Curves & F1-vs-Threshold Curves

Loads each model's `best.pt`, collects validation/test/external probabilities, and saves PR curves, F1-vs-threshold curves and the `curves_*.npz` arrays consumed by the threshold-selection cells.

In [ ]:
# ============================================================
# DISSERTATION PLOTS CELL (ALL 3 MODELS)
# - Loads best.pt from outputs_all_3_models_v2
# - Collects probabilities on VAL / TEST / EXT (skipping bad clips)
# - Plots and SAVES to Drive:
#     (1) Precision–Recall curve (with PR-AUC)
#     (2) F1 vs Threshold curve (and marks best-F1 threshold)
#
# OUTPUTS (per model, per split):
#   SAVE_ROOT/<model>/plots/pr_curve_<split>.png
#   SAVE_ROOT/<model>/plots/f1_curve_<split>.png
#   SAVE_ROOT/<model>/plots/curves_<split>.npz   (y, p1, thresholds)
#
# REQUIRES already defined:
#   device, NUM_CLASSES
#   val_df_m, test_df_m, external_df_m
#   NPZClipsDataset + _filter_bad_batch (robust versions)
#   build_tsm_fast, build_cnn_gru, build_r3d18
# ============================================================

import os
import numpy as np
import torch
from torch.utils.data import DataLoader

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

from tqdm import tqdm
from sklearn.metrics import (
    precision_recall_curve,
    average_precision_score,
    f1_score,
    precision_score,
    recall_score,
)

SAVE_ROOT = "/content/drive/MyDrive/ES327_Drowsiness/outputs_all_3_models_v2"
assert os.path.isdir(SAVE_ROOT), f"Missing SAVE_ROOT: {SAVE_ROOT}"

# -------- loader helper --------
def make_loader(df, mode, batch_size, num_workers=2):
    ds = NPZClipsDataset(df, mode=mode)
    return DataLoader(
        ds,
        batch_size=batch_size,
        shuffle=False,
        num_workers=num_workers,
        pin_memory=(device.type == "cuda"),
        persistent_workers=False,  # colab-safe
    )

def load_best(builder, ckpt_path):
    model = builder(NUM_CLASSES).to(device)
    ckpt = torch.load(ckpt_path, map_location=device)
    model.load_state_dict(ckpt["model_state"])
    model.eval()
    return model

def collect_probs(model, loader, desc=""):
    model.eval()
    ys, probs = [], []
    bad_total = 0

    it = loader
    if desc:
        it = tqdm(loader, desc=desc, leave=False)

    with torch.no_grad():
        for x, y, bad, paths in it:
            x, y, n_bad, _ = _filter_bad_batch(x, y, bad, paths)
            bad_total += n_bad
            if x is None:
                continue

            x = x.to(device, non_blocking=True).float() / 255.0
            y = y.to(device, non_blocking=True)

            logits = model(x)
            p1 = torch.softmax(logits, dim=1)[:, 1]  # P(y=1)
            ys.append(y.detach().cpu().numpy())
            probs.append(p1.detach().cpu().numpy())

    if len(ys) == 0:
        return np.array([], dtype=np.int64), np.array([], dtype=np.float32), bad_total

    return np.concatenate(ys), np.concatenate(probs), bad_total

def f1_vs_threshold(y_true, p1, thresholds):
    f1s = np.zeros_like(thresholds, dtype=np.float32)
    for i, t in enumerate(thresholds):
        y_pred = (p1 >= t).astype(np.int64)
        f1s[i] = f1_score(y_true, y_pred, zero_division=0)
    return f1s

def save_pr_curve(model_dir, split_name, y_true, p1):
    prec, rec, thr = precision_recall_curve(y_true, p1)
    ap = average_precision_score(y_true, p1)

    out_dir = os.path.join(model_dir, "plots")
    os.makedirs(out_dir, exist_ok=True)

    # PR curve
    plt.figure()
    plt.plot(rec, prec)
    plt.xlabel("Recall")
    plt.ylabel("Precision")
    plt.title(f"Precision–Recall ({split_name}) | PR-AUC={ap:.3f}")
    pr_path = os.path.join(out_dir, f"pr_curve_{split_name}.png")
    plt.savefig(pr_path, dpi=200, bbox_inches="tight")
    plt.close()

    # Save raw arrays
    np.savez(os.path.join(out_dir, f"curves_{split_name}.npz"),
             y=y_true.astype(np.int64), p1=p1.astype(np.float32),
             pr_precision=prec.astype(np.float32), pr_recall=rec.astype(np.float32), pr_thresholds=thr.astype(np.float32))

    return pr_path, ap

def save_f1_curve(model_dir, split_name, y_true, p1, grid_min=0.001, grid_max=0.999, step=0.001):
    # Dense threshold grid for a smooth curve (still fast)
    thresholds = np.arange(grid_min, grid_max + 1e-12, step, dtype=np.float32)
    f1s = f1_vs_threshold(y_true, p1, thresholds)

    best_i = int(np.argmax(f1s))
    best_thr = float(thresholds[best_i])
    best_f1 = float(f1s[best_i])

    out_dir = os.path.join(model_dir, "plots")
    os.makedirs(out_dir, exist_ok=True)

    plt.figure()
    plt.plot(thresholds, f1s)
    plt.xlabel("Threshold")
    plt.ylabel("F1")
    plt.title(f"F1 vs Threshold ({split_name}) | best F1={best_f1:.3f} @ thr={best_thr:.3f}")
    # mark best point
    plt.scatter([best_thr], [best_f1], s=25)
    f1_path = os.path.join(out_dir, f"f1_curve_{split_name}.png")
    plt.savefig(f1_path, dpi=200, bbox_inches="tight")
    plt.close()

    # Save raw arrays (append to same npz by saving separate file)
    np.savez(os.path.join(out_dir, f"f1_{split_name}.npz"),
             thresholds=thresholds.astype(np.float32), f1=f1s.astype(np.float32),
             best_thr=np.array([best_thr], dtype=np.float32),
             best_f1=np.array([best_f1], dtype=np.float32))

    return f1_path, best_thr, best_f1

def run_plots_for_model(name, builder, mode, bs, num_workers=2):
    model_dir = os.path.join(SAVE_ROOT, name)
    ckpt_path = os.path.join(model_dir, "best.pt")
    assert os.path.exists(ckpt_path), f"Missing checkpoint: {ckpt_path}"

    model = load_best(builder, ckpt_path)

    loaders = {
        "val": make_loader(val_df_m, mode=mode, batch_size=bs, num_workers=num_workers),
        "test": make_loader(test_df_m, mode=mode, batch_size=bs, num_workers=num_workers),
        "ext": make_loader(external_df_m, mode=mode, batch_size=bs, num_workers=num_workers),
    }

    print(f"\n==== PLOTTING: {name} ====")
    summary = {"model": name, "dir": model_dir, "splits": {}}

    for split_name, loader in loaders.items():
        y, p1, bad = collect_probs(model, loader, desc=f"{name} {split_name.upper()}")
        assert len(y) > 0, f"{name}: no samples for split {split_name}"

        pr_path, ap = save_pr_curve(model_dir, split_name, y, p1)
        f1_path, best_thr, best_f1 = save_f1_curve(model_dir, split_name, y, p1)

        # also compute precision/recall at best F1 threshold (for reference)
        y_pred = (p1 >= best_thr).astype(np.int64)
        prec = float(precision_score(y, y_pred, zero_division=0))
        rec  = float(recall_score(y, y_pred, zero_division=0))

        summary["splits"][split_name] = {
            "n": int(len(y)),
            "bad": int(bad),
            "prauc": float(ap),
            "best_f1": float(best_f1),
            "best_thr": float(best_thr),
            "best_prec": prec,
            "best_rec": rec,
            "pr_curve_png": pr_path,
            "f1_curve_png": f1_path,
        }

        print(f"  {split_name.upper():4s} | PR-AUC={ap:.3f} | bestF1={best_f1:.3f} @ thr={best_thr:.3f} | saved plots")

    # save summary json
    with open(os.path.join(model_dir, "plots", "plot_summary.json"), "w") as f:
        import json
        json.dump(summary, f, indent=2)

    print("✅ saved:", os.path.join(model_dir, "plots", "plot_summary.json"))
    return summary

# ---- run all 3 ----
RUNS = [
    ("tsm_fast", build_tsm_fast, "cthw", 32),
    ("cnn_gru",  build_cnn_gru,  "tchw", 16),
    ("r3d18",    build_r3d18,    "cthw", 12),
]

plot_summaries = []
for name, builder, mode, bs in RUNS:
    plot_summaries.append(run_plots_for_model(name, builder, mode, bs, num_workers=2))

print("\n✅ DONE. Open Drive ->", SAVE_ROOT, "-> each model -> plots/")

---
## 3. Dual-Threshold Selection (Validation Only)

Two complementary decision-threshold rules selected on validation data with no test leakage:

- **Primary** — maximise F1 subject to precision ≥ 0.60
- **Secondary** — maximise F1 subject to recall ≥ 0.70 (high-recall, safety-oriented)

The frozen thresholds are then applied to the test and external (DROZY) sets.

In [ ]:
# ============================================================
# STEP 2+3: Dual-threshold selection on VAL, then evaluate on TEST+EXT
# Uses saved curves_*_<split>.npz produced by your plotting cell (fast).
# Saves:
#   <SAVE_ROOT>/<model>/final_eval_dual_threshold.json
#   <SAVE_ROOT>/final_eval_dual_threshold_summary.csv
# ============================================================

import os, json
import numpy as np
import pandas as pd
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    average_precision_score, confusion_matrix
)

SAVE_ROOT = "/content/drive/MyDrive/ES327_Drowsiness/outputs_all_3_models_v2"
MODELS = ["tsm_fast", "cnn_gru", "r3d18"]

PREC_TARGET = 0.60   # balanced rule: maximise F1 with precision >= this
THR_MIN, THR_MAX, THR_STEP = 0.001, 0.999, 0.001

def load_split_npz(model_name, split):
    p = os.path.join(SAVE_ROOT, model_name, "plots", f"curves_{split}.npz")
    assert os.path.exists(p), f"Missing: {p} (run the plotting cell first)"
    z = np.load(p)
    y = z["y"].astype(np.int64)
    p1 = z["p1"].astype(np.float32)
    return y, p1

def metrics_at(y, p1, thr):
    yhat = (p1 >= thr).astype(np.int64)
    return {
        "thr": float(thr),
        "n": int(len(y)),
        "acc": float(accuracy_score(y, yhat)),
        "prec": float(precision_score(y, yhat, zero_division=0)),
        "rec": float(recall_score(y, yhat, zero_division=0)),
        "f1": float(f1_score(y, yhat, zero_division=0)),
        "prauc": float(average_precision_score(y, p1)),
        "cm": confusion_matrix(y, yhat).tolist(),
    }

def sweep_thresholds(y, p1, thresholds):
    # returns arrays for acc/prec/rec/f1
    f1s = np.zeros_like(thresholds, dtype=np.float32)
    precs = np.zeros_like(thresholds, dtype=np.float32)
    recs = np.zeros_like(thresholds, dtype=np.float32)
    accs = np.zeros_like(thresholds, dtype=np.float32)

    for i, t in enumerate(thresholds):
        yhat = (p1 >= t).astype(np.int64)
        accs[i]  = accuracy_score(y, yhat)
        precs[i] = precision_score(y, yhat, zero_division=0)
        recs[i]  = recall_score(y, yhat, zero_division=0)
        f1s[i]   = f1_score(y, yhat, zero_division=0)
    return accs, precs, recs, f1s

thresholds = np.arange(THR_MIN, THR_MAX + 1e-12, THR_STEP, dtype=np.float32)

all_rows = []
all_json = {}

for m in MODELS:
    yv, pv = load_split_npz(m, "val")
    yt, pt = load_split_npz(m, "test")
    ye, pe = load_split_npz(m, "ext")

    accs, precs, recs, f1s = sweep_thresholds(yv, pv, thresholds)

    # A) Best-F1 threshold on VAL
    i_best_f1 = int(np.argmax(f1s))
    thr_best_f1 = float(thresholds[i_best_f1])

    # B) Balanced threshold: max F1 with precision >= target
    ok = np.where(precs >= PREC_TARGET)[0]
    if len(ok) > 0:
        ok_best = ok[np.argmax(f1s[ok])]
        thr_bal = float(thresholds[int(ok_best)])
        bal_rule = f"maxF1 s.t. precision>={PREC_TARGET:.2f}"
    else:
        # fallback: max accuracy
        i_best_acc = int(np.argmax(accs))
        thr_bal = float(thresholds[i_best_acc])
        bal_rule = "fallback=maxAccuracy (precision target not achievable)"

    out = {
        "model": m,
        "precision_target": PREC_TARGET,
        "thr_best_f1_val": thr_best_f1,
        "thr_balanced_val": thr_bal,
        "balanced_rule": bal_rule,
        "VAL@bestF1": metrics_at(yv, pv, thr_best_f1),
        "TEST@bestF1": metrics_at(yt, pt, thr_best_f1),
        "EXT@bestF1": metrics_at(ye, pe, thr_best_f1),
        "VAL@balanced": metrics_at(yv, pv, thr_bal),
        "TEST@balanced": metrics_at(yt, pt, thr_bal),
        "EXT@balanced": metrics_at(ye, pe, thr_bal),
    }

    all_json[m] = out

    # Save per-model json
    out_path = os.path.join(SAVE_ROOT, m, "final_eval_dual_threshold.json")
    with open(out_path, "w") as f:
        json.dump(out, f, indent=2)
    print(f"✅ saved: {out_path}")

    # Rows for summary table
    def add_rows(tag):
        val = out[f"VAL@{tag}"]
        test = out[f"TEST@{tag}"]
        ext = out[f"EXT@{tag}"]
        all_rows.append({
            "model": m,
            "threshold_type": tag,
            "thr": val["thr"],
            "val_f1": val["f1"],
            "val_prauc": val["prauc"],
            "test_f1": test["f1"],
            "test_prauc": test["prauc"],
            "ext_f1": ext["f1"],
            "ext_prauc": ext["prauc"],
            "balanced_rule": out["balanced_rule"] if tag == "balanced" else "",
        })

    add_rows("bestF1")
    add_rows("balanced")

# Save summary CSV
df = pd.DataFrame(all_rows)
csv_path = os.path.join(SAVE_ROOT, "final_eval_dual_threshold_summary.csv")
df.to_csv(csv_path, index=False)
print(f"\n✅ saved summary CSV: {csv_path}")

# Display table
df

In [ ]:
# ============================================================
# Balanced threshold v2: max F1 subject to recall >= target (VAL only)
# Then evaluate VAL/TEST/EXT at:
#   - bestF1 threshold
#   - high-recall threshold
# Saves:
#   <SAVE_ROOT>/<model>/final_eval_dual_threshold_recall.json
#   <SAVE_ROOT>/final_eval_dual_threshold_recall_summary.csv
# ============================================================

import os, json
import numpy as np
import pandas as pd
from sklearn.metrics import (
    accuracy_score, precision_score, recall_score, f1_score,
    average_precision_score, confusion_matrix
)

SAVE_ROOT = "/content/drive/MyDrive/ES327_Drowsiness/outputs_all_3_models_v2"
MODELS = ["tsm_fast", "cnn_gru", "r3d18"]

RECALL_TARGET = 0.70
THR_MIN, THR_MAX, THR_STEP = 0.001, 0.999, 0.001
thresholds = np.arange(THR_MIN, THR_MAX + 1e-12, THR_STEP, dtype=np.float32)

def load_split_npz(model_name, split):
    p = os.path.join(SAVE_ROOT, model_name, "plots", f"curves_{split}.npz")
    assert os.path.exists(p), f"Missing: {p} (run plotting cell first)"
    z = np.load(p)
    y = z["y"].astype(np.int64)
    p1 = z["p1"].astype(np.float32)
    return y, p1

def metrics_at(y, p1, thr):
    yhat = (p1 >= thr).astype(np.int64)
    return {
        "thr": float(thr),
        "n": int(len(y)),
        "acc": float(accuracy_score(y, yhat)),
        "prec": float(precision_score(y, yhat, zero_division=0)),
        "rec": float(recall_score(y, yhat, zero_division=0)),
        "f1": float(f1_score(y, yhat, zero_division=0)),
        "prauc": float(average_precision_score(y, p1)),
        "cm": confusion_matrix(y, yhat).tolist(),
    }

def sweep(y, p1, thresholds):
    accs = np.zeros_like(thresholds, dtype=np.float32)
    precs= np.zeros_like(thresholds, dtype=np.float32)
    recs = np.zeros_like(thresholds, dtype=np.float32)
    f1s  = np.zeros_like(thresholds, dtype=np.float32)
    for i, t in enumerate(thresholds):
        yhat = (p1 >= t).astype(np.int64)
        accs[i]  = accuracy_score(y, yhat)
        precs[i] = precision_score(y, yhat, zero_division=0)
        recs[i]  = recall_score(y, yhat, zero_division=0)
        f1s[i]   = f1_score(y, yhat, zero_division=0)
    return accs, precs, recs, f1s

rows = []

for m in MODELS:
    yv, pv = load_split_npz(m, "val")
    yt, pt = load_split_npz(m, "test")
    ye, pe = load_split_npz(m, "ext")

    accs, precs, recs, f1s = sweep(yv, pv, thresholds)

    # bestF1
    i_best = int(np.argmax(f1s))
    thr_best = float(thresholds[i_best])

    # high-recall threshold: max F1 s.t. recall >= target
    ok = np.where(recs >= RECALL_TARGET)[0]
    if len(ok) > 0:
        ok_best = ok[np.argmax(f1s[ok])]
        thr_hr = float(thresholds[int(ok_best)])
        rule = f"maxF1 s.t. recall>={RECALL_TARGET:.2f}"
    else:
        # fallback: choose threshold with max recall (closest to target)
        i_maxrec = int(np.argmax(recs))
        thr_hr = float(thresholds[i_maxrec])
        rule = f"fallback=maxRecall (could not reach recall>={RECALL_TARGET:.2f})"

    out = {
        "model": m,
        "recall_target": RECALL_TARGET,
        "thr_best_f1_val": thr_best,
        "thr_high_recall_val": thr_hr,
        "rule": rule,
        "VAL@bestF1": metrics_at(yv, pv, thr_best),
        "TEST@bestF1": metrics_at(yt, pt, thr_best),
        "EXT@bestF1": metrics_at(ye, pe, thr_best),
        "VAL@highRecall": metrics_at(yv, pv, thr_hr),
        "TEST@highRecall": metrics_at(yt, pt, thr_hr),
        "EXT@highRecall": metrics_at(ye, pe, thr_hr),
    }

    save_path = os.path.join(SAVE_ROOT, m, "final_eval_dual_threshold_recall.json")
    with open(save_path, "w") as f:
        json.dump(out, f, indent=2)
    print(f"✅ saved: {save_path}")

    def add_row(tag):
        val = out[f"VAL@{tag}"]
        test = out[f"TEST@{tag}"]
        ext = out[f"EXT@{tag}"]
        rows.append({
            "model": m,
            "threshold_type": tag,
            "thr": val["thr"],
            "val_f1": val["f1"],
            "val_prec": val["prec"],
            "val_rec": val["rec"],
            "test_f1": test["f1"],
            "test_prec": test["prec"],
            "test_rec": test["rec"],
            "ext_f1": ext["f1"],
            "ext_prec": ext["prec"],
            "ext_rec": ext["rec"],
            "prauc_val": val["prauc"],
            "prauc_test": test["prauc"],
            "prauc_ext": ext["prauc"],
            "rule": out["rule"] if tag == "highRecall" else "",
        })

    add_row("bestF1")
    add_row("highRecall")

df2 = pd.DataFrame(rows)
csv_path = os.path.join(SAVE_ROOT, "final_eval_dual_threshold_recall_summary.csv")
df2.to_csv(csv_path, index=False)
print(f"\n✅ saved summary CSV: {csv_path}")
df2

---
## 4. Final Results Table (Table 4.2)

Assembles the per-dataset, per-model metrics into the final dissertation results table (CSV + JSON + PNG).

In [ ]:
# ============================================================
# FINAL DISSERTATION RESULTS TABLE (CSV + pretty PNG)
# - finds your latest "dual_threshold_recall_summary.csv" even if name/path differs
# - saves:
#   1) dissertation_final_results_table.csv
#   2) dissertation_final_results_table.json
#   3) dissertation_final_results_table.png
# ============================================================

from google.colab import drive
drive.mount("/content/drive", force_remount=False)

import os, glob, json
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

# ---- where your outputs live ----
ROOT = Path("/content/drive/MyDrive/ES327_Drowsiness")

# Prefer the v2 folder, but fall back if needed
PREFERRED = ROOT / "outputs_all_3_models_v2"
FALLBACKS = [
    ROOT / "outputs_all_3_models_v2",
    ROOT / "outputs_all_3_models",
    ROOT / "outputs_all_3_models_v3",
    ROOT,  # last resort (search anywhere under ES327_Drowsiness)
]

base = None
for b in FALLBACKS:
    if b.exists():
        base = b
        break

assert base is not None, f"Could not find outputs folder under {ROOT}"

print("✅ Searching under:", base)

# ---- find the summary CSV you generated ----
# Most likely:
#   outputs_all_3_models_v2/final_eval_dual_threshold_recall_summary.csv
# But we search robustly.
patterns = [
    str(base / "final_eval_dual_threshold_recall_summary.csv"),
    str(base / "**" / "final_eval_dual_threshold_recall_summary.csv"),
    str(base / "**" / "*dual_threshold*recall*summary*.csv"),
    str(base / "**" / "*dual_threshold*recall*.csv"),
    str(base / "**" / "*summary*.csv"),
]

cands = []
for pat in patterns:
    cands.extend(glob.glob(pat, recursive=True))

# de-dupe while preserving order
seen = set()
cands = [p for p in cands if not (p in seen or seen.add(p))]

assert len(cands) > 0, (
    "Couldn't find the recall summary CSV.\n"
    "Check Drive path and confirm the file exists in outputs_all_3_models_v2.\n"
    "Expected something like: final_eval_dual_threshold_recall_summary.csv"
)

# pick the most recently modified candidate
cands = sorted(cands, key=lambda p: os.path.getmtime(p), reverse=True)
IN_CSV = cands[0]
print("✅ Using:", IN_CSV)

df = pd.read_csv(IN_CSV)

# ---- Keep only one row per model ----
# Your CSV likely has both bestF1 & highRecall rows (sometimes identical);
# we keep bestF1 if present, else the first row.
if "threshold_type" in df.columns:
    if (df["threshold_type"] == "bestF1").any():
        df_one = df[df["threshold_type"] == "bestF1"].copy()
    else:
        df_one = df.copy()
else:
    df_one = df.copy()

# If still multiple rows per model, keep first per model
df_one = df_one.sort_values(list(df_one.columns)).groupby("model", as_index=False).first()

# ---- Rename models nicely ----
name_map = {
    "tsm_fast": "TSM-fast (Temporal Avg)",
    "cnn_gru":  "CNN-GRU (ResNet18+GRU)",
    "r3d18":    "3D ResNet-18",
}
df_one["model"] = df_one["model"].map(lambda x: name_map.get(x, x))

# ---- Select columns (handle both schemas) ----
# Your recall-summary CSV includes these:
preferred_cols = [
    "model", "thr",
    "val_f1", "val_prec", "val_rec",
    "test_f1", "test_prec", "test_rec",
    "ext_f1", "ext_prec", "ext_rec",
    "prauc_val", "prauc_test", "prauc_ext",
]
# Some earlier scripts used different PR-AUC column names
alt_map = {
    "val_prauc":  "prauc_val",
    "test_prauc": "prauc_test",
    "ext_prauc":  "prauc_ext",
}

for old, new in alt_map.items():
    if old in df_one.columns and new not in df_one.columns:
        df_one[new] = df_one[old]

cols = [c for c in preferred_cols if c in df_one.columns]
assert "model" in cols and "thr" in cols, "Missing required columns (model/thr)."

df_final = df_one[cols].copy()

# ---- Round for dissertation readability ----
for c in df_final.columns:
    if c != "model":
        df_final[c] = pd.to_numeric(df_final[c], errors="coerce")

round_map = {c: 3 for c in df_final.columns if c != "model"}
round_map["thr"] = 3
df_final = df_final.round(round_map)

# ---- Save outputs next to your outputs folder ----
# If base is ES327_Drowsiness, save into outputs_all_3_models_v2 if it exists.
out_dir = PREFERRED if PREFERRED.exists() else base
out_dir = Path(out_dir)

OUT_CSV  = out_dir / "dissertation_final_results_table.csv"
OUT_JSON = out_dir / "dissertation_final_results_table.json"
OUT_PNG  = out_dir / "dissertation_final_results_table.png"

df_final.to_csv(OUT_CSV, index=False)
with open(OUT_JSON, "w") as f:
    json.dump(df_final.to_dict(orient="records"), f, indent=2)

# ---- Render a nice PNG table ----
plt.figure(figsize=(18, 2.5 + 0.6 * len(df_final)))
plt.axis("off")

table = plt.table(
    cellText=df_final.values,
    colLabels=list(df_final.columns),
    loc="center",
    cellLoc="center",
)

table.auto_set_font_size(False)
table.set_fontsize(10)
table.scale(1, 1.35)

# bold header
for (r, c), cell in table.get_celld().items():
    if r == 0:
        cell.set_text_props(weight="bold")

plt.tight_layout()
plt.savefig(OUT_PNG, dpi=220, bbox_inches="tight")
plt.close()

print("\n✅ Saved:")
print(" -", OUT_CSV)
print(" -", OUT_JSON)
print(" -", OUT_PNG)

# show in notebook
df_final

---
## 5. Grad-CAM Saliency Analysis

Generates Grad-CAM heat-map grids for all three models using diagnostic case selection — the most confident true positive, most confident false positive, hardest false negative, and cleanest true negative — to show which facial regions drive each model's drowsiness decision (Appendix A.4).

In [ ]:
# =========================================================
# Grad-CAM for ALL 3 models using DIAGNOSTIC CASE SELECTION
# - most confident TP
# - most confident FP
# - hardest FN
# - cleanest TN
# Saves PNG + JSON to Drive
# =========================================================
import os, json, math
import numpy as np
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt

SAVE_ROOT = "/content/drive/MyDrive/ES327_Drowsiness/outputs_all_3_models_v2"
OUT_SUBDIR = "gradcam_diagnostic"
os.makedirs(SAVE_ROOT, exist_ok=True)

# ---------------------------------------------------------
# Helpers
# ---------------------------------------------------------
def ensure_dir(p):
    os.makedirs(p, exist_ok=True)

def load_ckpt(builder, ckpt_path):
    model = builder(NUM_CLASSES).to(device)
    ckpt = torch.load(ckpt_path, map_location=device)
    model.load_state_dict(ckpt["model_state"], strict=True)
    model.eval()
    return model

def to_numpy_img(x_chw):
    x = x_chw.astype(np.float32)
    if x.max() > 1.5:
        x = x / 255.0
    x = np.clip(x, 0, 1)
    return np.transpose(x, (1, 2, 0))  # CHW -> HWC

def overlay_heatmap(img_hwc, cam_hw, alpha=0.45):
    cam = np.clip(cam_hw, 0, 1)
    heat = plt.get_cmap("jet")(cam)[:, :, :3]
    out = (1 - alpha) * img_hwc + alpha * heat
    return np.clip(out, 0, 1)

def save_grid(frames_hwc, cams_hw, title, save_path, max_frames=8):
    T = len(frames_hwc)
    if T <= max_frames:
        idxs = list(range(T))
    else:
        idxs = np.linspace(0, T - 1, max_frames).round().astype(int).tolist()

    n = len(idxs)
    cols = min(n, 4)
    rows = math.ceil(n / cols)

    plt.figure(figsize=(4 * cols, 4 * rows))
    for i, t in enumerate(idxs):
        over = overlay_heatmap(frames_hwc[t], cams_hw[t])
        ax = plt.subplot(rows, cols, i + 1)
        ax.imshow(over)
        ax.set_title(f"t={t}")
        ax.axis("off")

    plt.suptitle(title, fontsize=14)
    plt.tight_layout()
    plt.savefig(save_path, dpi=200, bbox_inches="tight")
    plt.close()

def load_clip_from_df(df_m, idx, mode):
    ds = NPZClipsDataset(df_m, mode=mode)
    x, y, bad, path = ds[idx]
    if bool(bad.item()):
        return None
    return x.clone(), int(y.item()), path

@torch.no_grad()
def prob_one(model, x):
    xb = x.unsqueeze(0).to(device).float() / 255.0
    logits = model(xb)
    p1 = torch.softmax(logits, dim=1)[:, 1].item()
    return float(p1)

# ---------------------------------------------------------
# Diagnostic case selection
# ---------------------------------------------------------
from tqdm import tqdm

def find_diagnostic_cases(df_m, model, mode, thr, max_search=None):

    if max_search is None:
        max_search = len(df_m)

    best = {"TP":None, "FP":None, "FN":None, "TN":None}

    for idx in tqdm(range(min(len(df_m), max_search)), desc="Scanning test set"):
        item = load_clip_from_df(df_m, idx, mode)
        if item is None:
            continue

        x, y, path = item
        p1 = prob_one(model, x)
        yhat = 1 if p1 >= thr else 0

        record = {
            "idx": int(idx),
            "y": int(y),
            "p1": float(p1),
            "thr": float(thr),
            "path": path,
        }

        # TP: y=1, pred=1, choose highest p1
        if y == 1 and yhat == 1:
            if (best["TP"] is None) or (p1 > best["TP"]["p1"]):
                best["TP"] = record

        # FP: y=0, pred=1, choose highest p1
        elif y == 0 and yhat == 1:
            if (best["FP"] is None) or (p1 > best["FP"]["p1"]):
                best["FP"] = record

        # FN: y=1, pred=0, choose highest p1 below threshold
        elif y == 1 and yhat == 0:
            if (best["FN"] is None) or (p1 > best["FN"]["p1"]):
                best["FN"] = record

        # TN: y=0, pred=0, choose lowest p1
        elif y == 0 and yhat == 0:
            if (best["TN"] is None) or (p1 < best["TN"]["p1"]):
                best["TN"] = record

    return best

# ---------------------------------------------------------
# Stability helpers for CNN-GRU Grad-CAM
# ---------------------------------------------------------
def set_bn_eval(m):
    if isinstance(m, (torch.nn.BatchNorm1d, torch.nn.BatchNorm2d, torch.nn.BatchNorm3d)):
        m.eval()

def set_dropout_eval(m):
    if isinstance(m, torch.nn.Dropout):
        m.eval()

# ---------------------------------------------------------
# Grad-CAM for TSM-fast / CNN-GRU
# ---------------------------------------------------------
def gradcam_resnet18_video(model, x_cthw_or_tchw, mode, target_class=1, needs_rnn_train=False):
    """
    For:
      - TSM-fast
      - CNN-GRU

    Hooks final conv in ResNet18 backbone.
    For CNN-GRU, needs_rnn_train=True because cuDNN GRU backward can fail in eval mode.
    """
    conv = model.backbone[7][-1].conv2

    acts, grads = [], []

    def fwd_hook(m, inp, out):
        acts.append(out)

    def bwd_hook(m, gin, gout):
        grads.append(gout[0])

    h1 = conv.register_forward_hook(fwd_hook)
    h2 = conv.register_full_backward_hook(bwd_hook)

    was_training = model.training

    if needs_rnn_train:
        model.train()
        model.apply(set_bn_eval)
        model.apply(set_dropout_eval)
    else:
        model.eval()

    model.zero_grad(set_to_none=True)

    xb = x_cthw_or_tchw.unsqueeze(0).to(device).float() / 255.0
    logits = model(xb)
    score = logits[:, target_class].sum()
    score.backward()

    if was_training:
        model.train()
    else:
        model.eval()

    h1.remove()
    h2.remove()

    A = acts[0]   # (B*T, C, h, w) effectively
    G = grads[0]  # same

    w = G.mean(dim=(2, 3), keepdim=True)
    cam = (w * A).sum(dim=1)
    cam = F.relu(cam)

    if mode == "cthw":
        T = xb.shape[2]
        H, W = xb.shape[3], xb.shape[4]
        frames = xb[0].permute(1, 0, 2, 3).detach().cpu().numpy()  # (T,C,H,W)
    else:
        T = xb.shape[1]
        H, W = xb.shape[3], xb.shape[4]
        frames = xb[0].detach().cpu().numpy()  # (T,C,H,W)

    cam = cam.view(T, cam.shape[-2], cam.shape[-1]).unsqueeze(1)
    cam = F.interpolate(cam, size=(H, W), mode="bilinear", align_corners=False)
    cam = cam.squeeze(1)
    cam_np = cam.detach().cpu().numpy()

    cams = []
    for t in range(T):
        c = cam_np[t]
        c = c - c.min()
        c = c / (c.max() + 1e-8)
        cams.append(c)

    frames_hwc = [to_numpy_img(frames[t]) for t in range(T)]
    return frames_hwc, cams

# ---------------------------------------------------------
# Grad-CAM for r3d18 using tensor hook (safer)
# ---------------------------------------------------------
def gradcam_r3d18(model, x_cthw, target_class=1, layer="layer4"):
    """
    Safer Grad-CAM for torchvision r3d18 using tensor hook.
    layer='layer4' is coarser; layer='layer3' can be sharper.
    """
    acts, grads = [], []

    if layer == "layer4":
        conv = model.layer4[1].conv2
    elif layer == "layer3":
        conv = model.layer3[1].conv2
    else:
        raise ValueError("layer must be 'layer3' or 'layer4'")

    def fwd_hook(m, inp, out):
        acts.append(out)
        out.register_hook(lambda g: grads.append(g))

    h = conv.register_forward_hook(fwd_hook)

    model.eval()
    model.zero_grad(set_to_none=True)

    xb = x_cthw.unsqueeze(0).to(device).float() / 255.0
    logits = model(xb)
    score = logits[:, target_class].sum()
    score.backward()

    h.remove()

    A = acts[0]   # (B,C,T',H',W')
    G = grads[0]  # same

    w = G.mean(dim=(2, 3, 4), keepdim=True)
    cam = (w * A).sum(dim=1)
    cam = F.relu(cam)

    T, H, W = xb.shape[2], xb.shape[3], xb.shape[4]
    cam = cam.unsqueeze(1)
    cam = F.interpolate(cam, size=(T, H, W), mode="trilinear", align_corners=False)
    cam = cam.squeeze(1)[0]
    cam_np = cam.detach().cpu().numpy()

    cams = []
    for t in range(T):
        c = cam_np[t]
        c = c - c.min()
        c = c / (c.max() + 1e-8)
        cams.append(c)

    frames = xb[0].permute(1, 0, 2, 3).detach().cpu().numpy()  # (T,C,H,W)
    frames_hwc = [to_numpy_img(frames[t]) for t in range(T)]
    return frames_hwc, cams

# ---------------------------------------------------------
# Run Grad-CAM for one model using diagnostic cases
# ---------------------------------------------------------
def run_gradcam_for_model(model_name, builder, mode, thr, max_search=None, r3d_layer="layer4"):
    exp_dir = os.path.join(SAVE_ROOT, model_name)
    ckpt = os.path.join(exp_dir, "best.pt")
    assert os.path.exists(ckpt), f"Missing ckpt: {ckpt}"

    out_dir = os.path.join(exp_dir, OUT_SUBDIR)
    ensure_dir(out_dir)

    model = load_ckpt(builder, ckpt)

    cases = find_diagnostic_cases(
        df_m=test_df_m,
        model=model,
        mode=mode,
        thr=thr,
        max_search=max_search
    )

    print(f"\n[{model_name}] Selected cases:")
    for tag, info in cases.items():
        if info is None:
            print(f"  - {tag}: not found")
        else:
            print(f"  - {tag}: idx={info['idx']} | y={info['y']} | p1={info['p1']:.6f}")

    for tag, info in cases.items():
        if info is None:
            continue

        idx = info["idx"]
        item = load_clip_from_df(test_df_m, idx, mode)
        if item is None:
            continue

        x, y, path = item

        if model_name == "r3d18":
            frames_hwc, cams = gradcam_r3d18(
                model, x, target_class=1, layer=r3d_layer
            )
            title = f"{model_name}({r3d_layer}) | {tag} | y={y} p1={info['p1']:.3f} thr={thr:.3f}"
            save_name = f"{model_name}_{r3d_layer}_{tag.lower()}_idx{idx}.png"
        else:
            needs_rnn = (model_name == "cnn_gru")
            frames_hwc, cams = gradcam_resnet18_video(
                model, x, mode=mode, target_class=1, needs_rnn_train=needs_rnn
            )
            title = f"{model_name} | {tag} | y={y} p1={info['p1']:.3f} thr={thr:.3f}"
            save_name = f"{model_name}_{tag.lower()}_idx{idx}.png"

        save_path = os.path.join(out_dir, save_name)
        save_grid(frames_hwc, cams, title=title, save_path=save_path, max_frames=8)

        meta = {
            "model": model_name,
            "tag": tag,
            "idx": idx,
            "y": int(y),
            "p1": float(info["p1"]),
            "thr": float(thr),
            "path": path,
            "png": save_path
        }
        if model_name == "r3d18":
            meta["layer"] = r3d_layer

        with open(save_path.replace(".png", ".json"), "w") as f:
            json.dump(meta, f, indent=2)

        print(f"✅ saved {model_name} {tag} -> {save_path}")

# ---------------------------------------------------------
# Locked thresholds
# ---------------------------------------------------------
THR = {
    "tsm_fast": 0.010,
    "cnn_gru":  0.001,
    "r3d18":    0.004
}

# ---------------------------------------------------------
# RUN ALL 3
# ---------------------------------------------------------
run_gradcam_for_model(
    model_name="tsm_fast",
    builder=build_tsm_fast,
    mode="cthw",
    thr=THR["tsm_fast"],
    max_search=None  # full test set
)

run_gradcam_for_model(
    model_name="cnn_gru",
    builder=build_cnn_gru,
    mode="tchw",
    thr=THR["cnn_gru"],
    max_search=None  # full test set
)

run_gradcam_for_model(
    model_name="r3d18",
    builder=build_r3d18,
    mode="cthw",
    thr=THR["r3d18"],
    max_search=None,   # full test set
    r3d_layer="layer4" # change to "layer3" if you want sharper maps
)

print("\n✅ Done.")
print("Check:")
print("  /content/drive/MyDrive/ES327_Drowsiness/outputs_all_3_models_v2/<model>/gradcam_diagnostic/")